# Домашня робота №2. Завантаження даних та основи SQL

**Датасет:** UCI Adult/Census Income  

У цій роботі я завантажу дані UCI Adult у PostgreSQL, створю staging- та типізовану таблиці, перевірю якість даних і виконаю базовий аналіз за допомогою SQL.

In [4]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

In [5]:
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [6]:
# Завантажуємо UCI Adult з офіційного джерела

from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)

df = pd.concat(
    [adult.data.features, adult.data.targets],
    axis=1
)

print('Розмір датасету:', df.shape)
display(df.head())

Розмір датасету: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [7]:
# Зберігаємо отриманий датасет у CSV для подальшого COPY

csv_path = '/content/hw3_adult.csv'

df.to_csv(csv_path, index=False)

print('CSV створено:', csv_path)
print('Кількість рядків:', len(df))

CSV створено: /content/hw3_adult.csv
Кількість рядків: 48842


In [8]:
# Створюємо staging-таблицю для сирих даних

create_staging = """
DROP TABLE IF EXISTS hw3_adult_clean CASCADE;
DROP TABLE IF EXISTS hw3_adult_staging CASCADE;

CREATE TABLE hw3_adult_staging (
    age_raw TEXT,
    workclass_raw TEXT,
    fnlwgt_raw TEXT,
    education_raw TEXT,
    education_num_raw TEXT,
    marital_status_raw TEXT,
    occupation_raw TEXT,
    relationship_raw TEXT,
    race_raw TEXT,
    sex_raw TEXT,
    capital_gain_raw TEXT,
    capital_loss_raw TEXT,
    hours_per_week_raw TEXT,
    native_country_raw TEXT,
    income_raw TEXT
);
"""

with engine.begin() as conn:
    conn.execute(text(create_staging))

print("Staging-таблицю створено")

Staging-таблицю створено


In [9]:
# Завантажуємо CSV у staging-таблицю через COPY FROM STDIN

raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                """
                COPY hw3_adult_staging (
                    age_raw,
                    workclass_raw,
                    fnlwgt_raw,
                    education_raw,
                    education_num_raw,
                    marital_status_raw,
                    occupation_raw,
                    relationship_raw,
                    race_raw,
                    sex_raw,
                    capital_gain_raw,
                    capital_loss_raw,
                    hours_per_week_raw,
                    native_country_raw,
                    income_raw
                )
                FROM STDIN
                WITH (
                    FORMAT CSV,
                    HEADER TRUE,
                    DELIMITER ','
                )
                """,
                file
            )

    raw_conn.commit()

finally:
    raw_conn.close()

print("Дані завантажено через COPY FROM STDIN")

Дані завантажено через COPY FROM STDIN


In [10]:
# Перевіряємо кількість рядків після завантаження

row_count_query = """
SELECT COUNT(*) AS loaded_rows
FROM hw3_adult_staging;
"""

with engine.connect() as conn:
    row_count = pd.read_sql(text(row_count_query), conn)

display(row_count)

,loaded_rows
0,48842


## 2. Staging і типізація даних

У staging-таблиці всі сирі поля зберігаються як `TEXT`, щоб імпорт не падав через порожні значення або спеціальні маркери. Під час перенесення в `hw3_adult_clean` значення очищуються через `TRIM` і `NULLIF`, а `?` у `workclass`, `occupation` та `native_country` перетворюється на SQL `NULL`.

| Колонка | Семантика і формат джерела | Тип у clean | Правило очищення / обмеження |
|---|---|---|---|
| `person_id` | технічний ідентифікатор | `BIGINT IDENTITY` | `PRIMARY KEY` |
| `age` | вік, ціле число | `SMALLINT` | `NOT NULL`, 16–100 |
| `workclass` | тип зайнятості, категорія / `?` | `TEXT` | `?` і порожнє → `NULL` |
| `fnlwgt` | статистична вага, ціле число | `INTEGER` | `NOT NULL`, > 0 |
| `education` | рівень освіти, категорія | `TEXT` | `NOT NULL`, обрізання пробілів |
| `education_num` | числовий код освіти | `SMALLINT` | `NOT NULL`, 1–20 |
| `marital_status` | сімейний стан, категорія | `TEXT` | `NOT NULL` |
| `occupation` | професія, категорія / `?` | `TEXT` | `?` і порожнє → `NULL` |
| `relationship`, `race`, `sex` | категоріальні ознаки | `TEXT` | `NOT NULL`, обрізання пробілів |
| `capital_gain`, `capital_loss` | невід'ємні цілі суми | `INTEGER` | `NOT NULL`, >= 0 |
| `hours_per_week` | робочі години на тиждень | `SMALLINT` | `NOT NULL`, 1–168 |
| `native_country` | країна, категорія / `?` | `TEXT` | `?` і порожнє → `NULL` |
| `income` | цільова категорія `<=50K` / `>50K` | `TEXT` | прибираю пробіли й крапку в кінці тестових значень; `NOT NULL` |

Для цього датасету немає календарних дат або timestamp-подій, тому `DATE` і `TIMESTAMPTZ` не використовуються. `NUMERIC(p,s)` також не потрібен, оскільки грошові поля в Adult подані як цілі значення. `income` залишаю як категоріальний `TEXT`, щоб зберегти зрозумілі вихідні мітки класів.


In [11]:
# Створюємо типізовану таблицю Adult

create_clean_table = """
DROP TABLE IF EXISTS hw3_adult_clean CASCADE;

CREATE TABLE hw3_adult_clean (
    person_id BIGINT
        GENERATED BY DEFAULT AS IDENTITY
        PRIMARY KEY,

    age SMALLINT NOT NULL
        CHECK (age BETWEEN 16 AND 100),

    workclass TEXT,

    fnlwgt INTEGER NOT NULL
        CHECK (fnlwgt > 0),

    education TEXT NOT NULL,

    education_num SMALLINT NOT NULL
        CHECK (education_num BETWEEN 1 AND 20),

    marital_status TEXT NOT NULL,

    occupation TEXT,

    relationship TEXT NOT NULL,

    race TEXT NOT NULL,

    sex TEXT NOT NULL,

    capital_gain INTEGER NOT NULL
        CHECK (capital_gain >= 0),

    capital_loss INTEGER NOT NULL
        CHECK (capital_loss >= 0),

    hours_per_week SMALLINT NOT NULL
        CHECK (hours_per_week BETWEEN 1 AND 168),

    native_country TEXT,

    income TEXT NOT NULL
);
"""

with engine.begin() as conn:
    conn.execute(text(create_clean_table))

print("Типізовану таблицю створено")

Типізовану таблицю створено


In [12]:
# Переносимо дані зі staging у типізовану таблицю

insert_clean = """
INSERT INTO hw3_adult_clean (
    age,
    workclass,
    fnlwgt,
    education,
    education_num,
    marital_status,
    occupation,
    relationship,
    race,
    sex,
    capital_gain,
    capital_loss,
    hours_per_week,
    native_country,
    income
)
SELECT
    NULLIF(TRIM(age_raw), '')::SMALLINT,
    NULLIF(NULLIF(TRIM(workclass_raw), ''), '?'),
    NULLIF(TRIM(fnlwgt_raw), '')::INTEGER,
    NULLIF(TRIM(education_raw), ''),
    NULLIF(TRIM(education_num_raw), '')::SMALLINT,
    NULLIF(TRIM(marital_status_raw), ''),
    NULLIF(NULLIF(TRIM(occupation_raw), ''), '?'),
    NULLIF(TRIM(relationship_raw), ''),
    NULLIF(TRIM(race_raw), ''),
    NULLIF(TRIM(sex_raw), ''),
    NULLIF(TRIM(capital_gain_raw), '')::INTEGER,
    NULLIF(TRIM(capital_loss_raw), '')::INTEGER,
    NULLIF(TRIM(hours_per_week_raw), '')::SMALLINT,
    NULLIF(NULLIF(TRIM(native_country_raw), ''), '?'),
    NULLIF(REPLACE(TRIM(income_raw), '.', ''), '')
FROM hw3_adult_staging;
"""

with engine.begin() as conn:
    conn.execute(text(insert_clean))

print("Дані очищено та перенесено у hw3_adult_clean")

Дані очищено та перенесено у hw3_adult_clean


In [13]:
# Порівнюємо кількість рядків у staging та clean таблицях

count_query = """
SELECT
    (SELECT COUNT(*) FROM hw3_adult_staging) AS staging_rows,
    (SELECT COUNT(*) FROM hw3_adult_clean) AS clean_rows;
"""

with engine.connect() as conn:
    counts = pd.read_sql(text(count_query), conn)

display(counts)

,staging_rows,clean_rows
0,48842,48842


### Контроль кількості рядків

У staging- та clean-таблицях по 48 842 рядки. Кількість не змінилася, тому що під час очищення я не відфільтровувала записи: спеціальні маркери та порожні значення нормалізуються до `NULL`, а валідні числові поля лише приводяться до потрібних типів.


## 3. Аудит якості даних

Після завантаження та типізації даних перевіряю їх якість за допомогою SQL-запитів. Спочатку перевірю кількість рядків, можливі дублікати та пропущені значення, а потім проаналізую категорії й числові ознаки.

In [14]:
# Перевіряємо повні дублікати в даних

duplicate_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        age,
        workclass,
        fnlwgt,
        education,
        education_num,
        marital_status,
        occupation,
        relationship,
        race,
        sex,
        capital_gain,
        capital_loss,
        hours_per_week,
        native_country,
        income
    )) AS duplicate_rows
FROM hw3_adult_clean;
"""

with engine.connect() as conn:
    duplicates = pd.read_sql(text(duplicate_query), conn)

display(duplicates)

,total_rows,duplicate_rows
0,48842,52


### Перевірка дублікатів

У таблиці 48 842 рядки. Перевірка без урахування технічного `person_id` показала 52 потенційні повні дублікати.

Оскільки набір Adult не містить гарантованого природного унікального ідентифікатора людини, однакові значення всіх ознак не обов'язково означають помилкове дублювання. Тому на цьому етапі я їх не видаляю, а лише фіксую як потенційні дублікати.


In [15]:
# Перевіряємо пропущені значення у важливих колонках

null_query = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (WHERE workclass IS NULL) AS workclass_nulls,
    ROUND(
        COUNT(*) FILTER (WHERE workclass IS NULL) * 100.0 / COUNT(*), 2
    ) AS workclass_null_pct,

    COUNT(*) FILTER (WHERE occupation IS NULL) AS occupation_nulls,
    ROUND(
        COUNT(*) FILTER (WHERE occupation IS NULL) * 100.0 / COUNT(*), 2
    ) AS occupation_null_pct,

    COUNT(*) FILTER (WHERE native_country IS NULL) AS native_country_nulls,
    ROUND(
        COUNT(*) FILTER (WHERE native_country IS NULL) * 100.0 / COUNT(*), 2
    ) AS native_country_null_pct,

    COUNT(*) FILTER (WHERE education IS NULL) AS education_nulls,
    COUNT(*) FILTER (WHERE hours_per_week IS NULL) AS hours_per_week_nulls

FROM hw3_adult_clean;
"""

with engine.connect() as conn:
    null_stats = pd.read_sql(text(null_query), conn)

display(null_stats)

,total_rows,workclass_nulls,workclass_null_pct,occupation_nulls,occupation_null_pct,native_country_nulls,native_country_null_pct,education_nulls,hours_per_week_nulls
0,48842,2799,5.73,2809,5.75,857,1.75,0,0


### Перевірка пропущених значень

Найбільше пропущених значень виявлено в `occupation` - 2 809 (5,75%) та `workclass` - 2 799 (5,73%). У `native_country` знайдено 857 пропусків (1,75%).

У колонках `education` та `hours_per_week` пропущених значень немає. Під час очищення спеціальні маркери пропусків були перетворені на SQL `NULL`, тому їх можна коректно враховувати в подальших SQL-запитах.

In [16]:
# Аналізуємо розподіл категорій workclass

workclass_query = """
SELECT
    COALESCE(workclass, 'Missing') AS workclass,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM hw3_adult_clean
GROUP BY workclass
ORDER BY count DESC;
"""

with engine.connect() as conn:
    workclass_stats = pd.read_sql(text(workclass_query), conn)

display(workclass_stats)

,workclass,count,percentage
0,Private,33906,69.42
1,Self-emp-not-inc,3862,7.91
2,Local-gov,3136,6.42
3,Missing,2799,5.73
4,State-gov,1981,4.06
5,Self-emp-inc,1695,3.47
6,Federal-gov,1432,2.93
7,Without-pay,21,0.04
8,Never-worked,10,0.02


### Розподіл за типом зайнятості

Найбільша категорія `workclass` - `Private`: 33 906 записів, або 69,42% датасету. Далі йдуть `Self-emp-not-inc` (7,91%) та `Local-gov` (6,42%).

Пропущені значення становлять 5,73%. Розподіл категорій нерівномірний, оскільки більшість записів належить до приватного сектору.

In [29]:
# Аналізуємо основні числові ознаки

numeric_stats_query = """
SELECT
    ROUND(AVG(age), 2) AS avg_age,
    MIN(age) AS min_age,
    MAX(age) AS max_age,
    ROUND(STDDEV_SAMP(age), 2) AS sd_age,

    ROUND(AVG(hours_per_week), 2) AS avg_hours_per_week,
    MIN(hours_per_week) AS min_hours_per_week,
    MAX(hours_per_week) AS max_hours_per_week,
    ROUND(STDDEV_SAMP(hours_per_week), 2) AS sd_hours_per_week,

    ROUND(AVG(capital_gain), 2) AS avg_capital_gain,
    MAX(capital_gain) AS max_capital_gain,
    ROUND(STDDEV_SAMP(capital_gain), 2) AS sd_capital_gain,

    ROUND(AVG(capital_loss), 2) AS avg_capital_loss,
    MAX(capital_loss) AS max_capital_loss,
    ROUND(STDDEV_SAMP(capital_loss), 2) AS sd_capital_loss

FROM hw3_adult_clean;
"""

numeric_stats = pd.read_sql(numeric_stats_query, engine)
display(numeric_stats)

,avg_age,min_age,max_age,sd_age,avg_hours_per_week,min_hours_per_week,max_hours_per_week,sd_hours_per_week,avg_capital_gain,max_capital_gain,sd_capital_gain,avg_capital_loss,max_capital_loss,sd_capital_loss
0,38.64,17,90,13.71,40.42,1,99,12.39,1079.07,99999,7452.02,87.5,4356,403.0


### Аналіз числових ознак

Середній вік у датасеті становить 38,64 року, а значення змінюються від 17 до 90 років. Стандартне відхилення віку становить 13,71 року.

У середньому люди працюють 40,42 години на тиждень. Мінімальне значення становить 1 годину, максимальне - 99 годин, а стандартне відхилення - 12,39.

Для `capital_gain` і `capital_loss` максимальні значення значно перевищують середні, а стандартні відхилення є великими відносно середніх значень. Це може вказувати на нерівномірний розподіл і наявність рідкісних великих значень, тому такі показники варто додатково перевірити під час аналізу аномалій.

In [18]:
# Перевіряємо записи з дуже великою кількістю робочих годин

outlier_query = """
SELECT
    person_id,
    age,
    workclass,
    occupation,
    hours_per_week,
    income
FROM hw3_adult_clean
WHERE hours_per_week > 80
ORDER BY hours_per_week DESC
LIMIT 20;
"""

with engine.connect() as conn:
    outliers = pd.read_sql(text(outlier_query), conn)

display(outliers)

,person_id,age,workclass,occupation,hours_per_week,income
0,10269,56,Self-emp-inc,Transport-moving,99,<=50K
1,8658,30,Private,Protective-serv,99,<=50K
2,8800,39,Private,Transport-moving,99,>50K
3,10146,35,None,None,99,<=50K
4,8397,50,Self-emp-inc,Exec-managerial,99,>50K
5,5436,44,Private,Prof-specialty,99,<=50K
6,5380,43,Private,Craft-repair,99,<=50K
7,8076,44,Self-emp-not-inc,Other-service,99,<=50K
8,9814,28,Private,Prof-specialty,99,<=50K
9,9834,67,Private,Prof-specialty,99,<=50K


### Перевірка потенційних аномалій

У датасеті є записи, де `hours_per_week` перевищує 80 годин, а максимальне значення становить 99 годин на тиждень. Такі значення виглядають нетипово, але можуть бути реальними.

Тому я не видаляю ці записи автоматично. Їх варто враховувати як потенційні викиди під час подальшого аналізу або побудови моделі.

## 4. DQL та EDA-запити

Нижче виконую SQL-запити для дослідницького аналізу даних. Для кожного запиту формулюю питання, виконую SQL та коротко аналізую отриманий результат.

### Запит 1. Люди віком від 25 до 40 років із повним робочим тижнем

**Питання:** Які записи відповідають людям віком від 25 до 40 років, які працюють не менше 40 годин на тиждень?

In [19]:
# Запит 1: фільтрація за віком та кількістю робочих годин

query_1 = """
SELECT
    person_id,
    age,
    workclass,
    occupation,
    hours_per_week,
    income
FROM hw3_adult_clean
WHERE age BETWEEN 25 AND 40
  AND hours_per_week >= 40
ORDER BY age, hours_per_week DESC
LIMIT 10;
"""

with engine.connect() as conn:
    result_1 = pd.read_sql(text(query_1), conn)

display(result_1)

,person_id,age,workclass,occupation,hours_per_week,income
0,1173,25,Private,Farming-fishing,99,>50K
1,15182,25,Private,Other-service,99,<=50K
2,15010,25,Private,Farming-fishing,96,<=50K
3,28480,25,Private,Handlers-cleaners,95,<=50K
4,31109,25,None,None,90,<=50K
5,4741,25,Private,Exec-managerial,84,>50K
6,23397,25,Private,Tech-support,84,<=50K
7,26146,25,None,None,80,<=50K
8,15376,25,Self-emp-not-inc,Craft-repair,80,<=50K
9,5511,25,Private,Craft-repair,80,>50K


**Висновок:** Запит показав людей віком від 25 до 40 років, які працюють щонайменше 40 годин на тиждень. Через сортування за віком та кількістю робочих годин серед перших результатів бачимо 25-річних із найбільшим робочим навантаженням.

У вибірці є значення від 80 до 99 годин на тиждень, що ще раз підтверджує наявність дуже високих значень `hours_per_week`.

### Запит 2. Розподіл за рівнем доходу

**Питання:** Яка кількість та частка людей належить до кожної категорії доходу?

In [20]:
# Запит 2: розподіл за категоріями доходу

query_2 = """
SELECT
    income,
    COUNT(*) AS people_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM hw3_adult_clean
GROUP BY income
ORDER BY people_count DESC;
"""

with engine.connect() as conn:
    result_2 = pd.read_sql(text(query_2), conn)

display(result_2)

,income,people_count,percentage
0,<=50K,37155,76.07
1,>50K,11687,23.93


**Висновок:** Більшість записів у датасеті належить до категорії доходу `<=50K` - 37 155 осіб (76,07%). Категорія `>50K` містить 11 687 записів (23,93%).

Отже, цільова ознака розподілена нерівномірно: людей із доходом `<=50K` приблизно втричі більше.

### Запит 3. Унікальні рівні освіти

**Питання:** Які унікальні рівні освіти представлені в датасеті?

In [21]:
# Запит 3: унікальні значення рівня освіти

query_3 = """
SELECT DISTINCT
    education,
    education_num
FROM hw3_adult_clean
ORDER BY education_num DESC;
"""

with engine.connect() as conn:
    result_3 = pd.read_sql(text(query_3), conn)

display(result_3)

,education,education_num
0,Doctorate,16
1,Prof-school,15
2,Masters,14
3,Bachelors,13
4,Assoc-acdm,12
5,Assoc-voc,11
6,Some-college,10
7,HS-grad,9
8,12th,8
9,11th,7


**Висновок:** У датасеті представлено 16 унікальних рівнів освіти. Значення `education_num` відповідає рівню освіти та змінюється від 1 для `Preschool` до 16 для `Doctorate`.

Таким чином, `education` містить текстову категорію, а `education_num` - її числове представлення.

### Запит 4. Дохід серед людей із вищими рівнями освіти

**Питання:** Як розподіляється рівень доходу серед людей із рівнями освіти Bachelors, Masters, Prof-school та Doctorate?

In [22]:
# Запит 4: дохід для вибраних рівнів освіти

query_4 = """
SELECT
    education,
    income,
    COUNT(*) AS people_count
FROM hw3_adult_clean
WHERE education IN (
    'Bachelors',
    'Masters',
    'Prof-school',
    'Doctorate'
)
GROUP BY education, income
ORDER BY education, people_count DESC;
"""

with engine.connect() as conn:
    result_4 = pd.read_sql(text(query_4), conn)

display(result_4)

,education,income,people_count
0,Bachelors,<=50K,4712
1,Bachelors,>50K,3313
2,Doctorate,>50K,431
3,Doctorate,<=50K,163
4,Masters,>50K,1459
5,Masters,<=50K,1198
6,Prof-school,>50K,617
7,Prof-school,<=50K,217


**Висновок:** Серед людей із рівнем освіти `Bachelors` більше записів із доходом `<=50K` - 4 712 проти 3 313 із доходом `>50K`.

Для `Masters`, `Prof-school` та `Doctorate` у вибірці, навпаки, кількість записів із доходом `>50K` більша. Наприклад, серед людей із `Doctorate` таких записів 431, а з доходом `<=50K` - 163.

### Запит 5. Професії, пов'язані з управлінням

**Питання:** Які записи містять професії, назви яких пов'язані з керуванням або менеджментом?

In [23]:
# Запит 5: пошук професій за частиною назви

query_5 = """
SELECT
    person_id,
    age,
    occupation,
    education,
    hours_per_week,
    income
FROM hw3_adult_clean
WHERE occupation IS NOT NULL
  AND (
      occupation ILIKE '%manager%'
      OR occupation ILIKE '%exec%'
  )
ORDER BY age DESC
LIMIT 10;
"""

with engine.connect() as conn:
    result_5 = pd.read_sql(text(query_5), conn)

display(result_5)

,person_id,age,occupation,education,hours_per_week,income
0,39976,90,Exec-managerial,Assoc-acdm,45,>50K
1,18834,90,Exec-managerial,Masters,40,<=50K
2,12977,90,Exec-managerial,10th,40,<=50K
3,24044,90,Exec-managerial,HS-grad,12,<=50K
4,15894,90,Exec-managerial,Bachelors,40,>50K
5,11999,90,Exec-managerial,Bachelors,55,<=50K
6,5374,90,Exec-managerial,Masters,60,>50K
7,1938,90,Exec-managerial,Bachelors,45,<=50K
8,5410,90,Exec-managerial,Masters,50,>50K
9,21836,88,Exec-managerial,Prof-school,40,<=50K


**Висновок:** Пошук за частиною назви професії знайшов записи категорії `Exec-managerial`. У перших 10 результатах вік становить від 88 до 90 років, оскільки дані відсортовані за віком у порядку спадання.

Серед цих записів є люди з обома категоріями доходу, тому сама належність до управлінської професії не означає однаковий рівень доходу.

### Запит 6. Записи з відсутньою професією

**Питання:** Які записи мають пропущене значення `occupation` і як їх можна зручно показати в результаті?

In [24]:
# Запит 6: записи з NULL у occupation

query_6 = """
SELECT
    person_id,
    age,
    COALESCE(workclass, 'Missing') AS workclass,
    COALESCE(occupation, 'Missing') AS occupation,
    education,
    income
FROM hw3_adult_clean
WHERE occupation IS NULL
ORDER BY age DESC
LIMIT 10;
"""

with engine.connect() as conn:
    result_6 = pd.read_sql(text(query_6), conn)

display(result_6)

,person_id,age,workclass,occupation,education,income
0,44433,90,Missing,Missing,10th,<=50K
1,4114,90,Missing,Missing,Bachelors,<=50K
2,24239,90,Missing,Missing,1st-4th,<=50K
3,31697,90,Missing,Missing,HS-grad,>50K
4,12454,90,Missing,Missing,Some-college,<=50K
5,25304,90,Missing,Missing,7th-8th,<=50K
6,11734,90,Missing,Missing,HS-grad,<=50K
7,8966,90,Missing,Missing,HS-grad,<=50K
8,47970,89,Missing,Missing,10th,<=50K
9,31433,87,Missing,Missing,HS-grad,<=50K


**Висновок:** Запит показав записи, у яких значення `occupation` відсутнє. За допомогою `COALESCE` SQL `NULL` відображається як `Missing`, що робить результат зручнішим для аналізу.

У перших 10 записах одночасно відсутні `occupation` і `workclass`. При цьому значення `education` та `income` у цих рядках збережені.

### Запит 7. Різниця у пропусках workclass та occupation

**Питання:** Які записи мають пропуск лише в одному з двох полів - `workclass` або `occupation`?


In [25]:
# Запит 7: порівнюємо наявність NULL у двох колонках

query_7 = """
SELECT
    person_id,
    age,
    workclass,
    occupation,
    income
FROM hw3_adult_clean
WHERE (workclass IS NULL) IS DISTINCT FROM (occupation IS NULL)
ORDER BY age DESC
LIMIT 20;
"""

with engine.connect() as conn:
    result_7 = pd.read_sql(text(query_7), conn)

display(result_7)

,person_id,age,workclass,occupation,income
0,32305,30,Never-worked,None,<=50K
1,10848,23,Never-worked,None,<=50K
2,23233,20,Never-worked,None,<=50K
3,44169,20,Never-worked,None,<=50K
4,46460,18,Never-worked,None,<=50K
5,20338,18,Never-worked,None,<=50K
6,32315,18,Never-worked,None,<=50K
7,5365,18,Never-worked,None,<=50K
8,14774,17,Never-worked,None,<=50K
9,41347,17,Never-worked,None,<=50K


**Висновок:** Запит знайшов записи, де наявність пропусків у `workclass` та `occupation` відрізняється. У показаних результатах `workclass` має значення `Never-worked`, а `occupation` є `NULL`.

Це виглядає логічно, оскільки для людини, яка ніколи не працювала, професія може бути не вказана. Тому такі `NULL` не обов'язково є помилкою в даних.

### Запит 8. Перетворення спеціального маркера на NULL

**Питання:** Скільки спеціальних значень `?` у сирих даних `workclass` перетворюються на `NULL` за допомогою `NULLIF`?

In [26]:
# Запит 8: перевіряємо різні види пропущених значень

query_8 = """
SELECT
    COUNT(*) FILTER (
        WHERE workclass_raw IS NULL
    ) AS original_nulls,

    COUNT(*) FILTER (
        WHERE TRIM(workclass_raw) = '?'
    ) AS question_mark_count,

    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(workclass_raw), '?') IS NULL
    ) AS null_after_nullif
FROM hw3_adult_staging;
"""

with engine.connect() as conn:
    result_8 = pd.read_sql(text(query_8), conn)

display(result_8)

,original_nulls,question_mark_count,null_after_nullif
0,963,1836,2799


**Висновок:** У сирій колонці `workclass_raw` знайдено 1 836 значень `?`. Після застосування `NULLIF(TRIM(workclass_raw), '?')` кількість `NULL` становить 2 799.

Різниця виникає тому, що частина значень уже була `NULL`, а `NULLIF` додатково перетворює маркер `?` на `NULL`. Це показує, що під час очищення потрібно враховувати різні способи представлення пропущених значень.

### Запит 9. Нетипові умови зайнятості

**Питання:** Які люди працюють не в основних категоріях приватної або самозайнятої роботи та при цьому працюють понад 60 годин на тиждень або мають додатний `capital_gain`?

In [27]:
# Запит 9: використання NOT, IN, AND та OR

query_9 = """
SELECT
    person_id,
    age,
    workclass,
    occupation,
    hours_per_week,
    capital_gain,
    income
FROM hw3_adult_clean
WHERE NOT (
    COALESCE(workclass, 'Missing') IN (
        'Private',
        'Self-emp-not-inc',
        'Self-emp-inc'
    )
)
AND (
    hours_per_week > 60
    OR capital_gain > 0
)
ORDER BY hours_per_week DESC, capital_gain DESC
LIMIT 10;
"""

with engine.connect() as conn:
    result_9 = pd.read_sql(text(query_9), conn)

display(result_9)

,person_id,age,workclass,occupation,hours_per_week,capital_gain,income
0,29990,43,Local-gov,Tech-support,99,4386,>50K
1,47377,49,None,None,99,4386,>50K
2,47108,32,State-gov,Prof-specialty,99,2961,<=50K
3,28113,61,None,None,99,0,<=50K
4,21057,64,Local-gov,Craft-repair,99,0,<=50K
5,12790,24,State-gov,Prof-specialty,99,0,<=50K
6,10146,35,None,None,99,0,<=50K
7,25807,49,None,None,99,0,<=50K
8,4091,50,None,None,99,0,<=50K
9,19733,34,Federal-gov,Prof-specialty,99,0,<=50K


**Висновок:** Запит показав людей, які не належать до основних категорій приватної або самозайнятої роботи та мають високе робоче навантаження або додатний `capital_gain`.

У перших результатах є працівники `Local-gov`, `State-gov`, а також записи з пропущеним `workclass`. Найбільше значення `hours_per_week` у цій вибірці становить 99 годин.

### Запит 10. Порівняння числових показників за рівнем доходу

**Питання:** Чи відрізняються середній вік та середня кількість робочих годин між групами доходу `<=50K` і `>50K`?

In [28]:
# Запит 10: порівнюємо середні показники за рівнем доходу

query_10 = """
SELECT
    income,
    COUNT(*) AS people_count,
    ROUND(AVG(age), 2) AS avg_age,
    ROUND(AVG(hours_per_week), 2) AS avg_hours_per_week
FROM hw3_adult_clean
GROUP BY income
ORDER BY income;
"""

with engine.connect() as conn:
    result_10 = pd.read_sql(text(query_10), conn)

display(result_10)

,income,people_count,avg_age,avg_hours_per_week
0,<=50K,37155,36.87,38.84
1,>50K,11687,44.28,45.45


**Висновок:** У групі з доходом `>50K` середній вік становить 44,28 року, а середня тривалість роботи - 45,45 години на тиждень.

Для групи `<=50K` ці показники нижчі: середній вік становить 36,87 року, а середня тривалість роботи - 38,84 години. Отже, у цьому датасеті група з доходом `>50K` у середньому старша та працює більше годин на тиждень.

## 5. Reflection

Під час перевірки якості даних я виявила кілька типових проблем. У категоріальних колонках `workclass`, `occupation` і `native_country` є пропущені значення та спеціальний маркер `?`, який потрібно перетворювати на SQL `NULL`. Також перевірка без технічного `person_id` показала 52 потенційні повні дублікати. Я не видаляла їх автоматично, тому що Adult не має гарантованого природного ідентифікатора людини. Окремо я побачила дуже великі значення `hours_per_week`, до 99 годин, але залишила їх як можливі рідкісні випадки.

Для ML-моделі прогнозування доходу я використала б `age`, `workclass`, `education` або `education_num`, `marital_status`, `occupation`, `relationship`, `hours_per_week`, `capital_gain`, `capital_loss` та частину інших категоріальних ознак після кодування. Я не використовувала б `person_id`, тому що це лише технічний ключ. `fnlwgt` я б спочатку окремо дослідила, оскільки це статистична вага вибірки, а не звичайна характеристика людини. Також не варто одночасно використовувати `education` і `education_num`, бо вони описують майже ту саму інформацію.

Очевидного leakage у вибраних ознаках я не бачу, якщо прогноз робиться за наявними характеристиками людини, а `income` використовується тільки як target. Для покращення pipeline я б запропонувала перевіряти допустимі категорії, автоматично нормалізувати всі sentinel-значення, контролювати дублікати та формувати звіт про якість після кожного завантаження. Природна ML-задача для Adult - бінарна classification: прогноз, чи належить дохід до класу `>50K`. Такий target уже має два класи, тому задача природно формулюється як класифікація, а не regression або forecasting.
